# Assignment 2: Milestone I Natural Language Processing
## Task 2&3
#### Student Name: Shiv Mahesh Bhosale
#### Student ID: S4192953


Environment: Python 397 and Jupyter notebook

Libraries used: please include all the libraries you used in your assignment, e.g.,:
* pandas
* re
* numpy

## Introduction
You should give a brief information of this assessment task here.

<span style="color: red"> Note that this is a sample notebook only. You will need to fill in the proper markdown and code blocks. You might also want to make necessary changes to the structure to meet your own needs. Note also that any generic comments written in this notebook are to be removed and replace with your own words.</span>

## Importing libraries 

In [1]:
import pandas as pd
import numpy as np
import regex as re
from collections import Counter
from itertools import chain
import gensim
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import lil_matrix
import warnings
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

import logging

import gensim.downloader as api
warnings.filterwarnings('ignore')
import os

In [2]:
# !pip install pandas numpy scikit-learn gensim nltk

In [3]:
# import sys

# print(sys.executable)
# print(sys.version)

In [4]:
# import sys
# !{sys.executable} -m pip install pandas numpy scikit-learn gensim nltk

In [5]:
import pandas as pd
import numpy as np
from collections import Counter
from itertools import chain
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import gensim.downloader as api
import urllib.request
import zipfile

## Task 2. Generating Feature Representations for Clothing Items Reviews

...... Sections and code blocks on buidling different document feature represetations


<span style="color: red"> You might have complex notebook structure in this section, please feel free to create your own notebook structure. </span>

In [6]:
def load_vocab(filepath='vocab'):
    vocab = {}
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if ':' in line:
                word, idx = line.rsplit(':', 1)
                vocab[word] = int(idx)
    return vocab

In [7]:
vocab = load_vocab('vocab.txt')
index_to_word = {idx: word for word, idx in vocab.items()}

In [8]:
print(f"Vocabulary size: {len(vocab)}")
print(f"\nFirst 10 vocab entries:")
for word, idx in list(vocab.items())[:10]:
    print(f"  {word}:{idx}")

Vocabulary size: 7529

First 10 vocab entries:
  a-cup:0
  a-flutter:1
  a-frame:2
  a-kind:3
  a-line:4
  a-lines:5
  a-symmetric:6
  aa:7
  ab:8
  abbey:9


In [9]:
df = pd.read_csv('processed.csv')

In [10]:
df

,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,1077,60,Some major design flaws,high hopes wanted work initially petite usual ...,3,0,0,General,Dresses,Dresses
1,1049,50,My favorite buy!,jumpsuit fun flirty fabulous time compliments,5,1,0,General Petite,Bottoms,Pants
2,847,47,Flattering shirt,shirt due adjustable front tie length leggings...,5,1,6,General,Tops,Blouses
3,1080,49,Not for the very petite,tracy reese dresses petite feet tall brand pre...,2,0,4,General,Dresses,Dresses
4,858,39,Cagrcoal shimmer fun,basket hte person store pick teh pale hte gorg...,5,1,1,General Petite,Tops,Knits
...,...,...,...,...,...,...,...,...,...,...
19657,1104,34,Great dress for many occasions,happy snag price easy slip cut combo,5,1,0,General Petite,Dresses,Dresses
19658,862,48,Wish it was made of cotton,reminds maternity clothes stretchy shiny mater...,3,1,0,General Petite,Tops,Knits
19659,1104,31,"Cute, but see through",worked glad store order online,3,0,1,General Petite,Dresses,Dresses
19660,1084,28,"Very cute dress, perfect for summer parties an...",wedding summer medium waist perfectly long big...,3,1,2,General,Dresses,Dresses


In [11]:
df["Review Text"].isnull().sum()

np.int64(10)

In [12]:
df['Review Text'] = df['Review Text'].fillna('')

In [13]:
tk_reviews = [review.split(' ') for review in df['Review Text'].tolist()]

In [14]:
tk_reviews[1]

['jumpsuit', 'fun', 'flirty', 'fabulous', 'time', 'compliments']

In [15]:
joined_reviews = [' '.join(tokens) for tokens in tk_reviews]

In [16]:
joined_reviews[1]

'jumpsuit fun flirty fabulous time compliments'

In [17]:
len(tk_reviews)

19662

In [18]:
cVectorizer = CountVectorizer(analyzer="word", vocabulary=vocab)
count_features = cVectorizer.fit_transform(joined_reviews)

In [19]:
count_features.shape

(19662, 7529)

In [20]:
out_file = open('count_vectors.txt', 'w')

In [21]:
for doc_id in range(count_features.shape[0]):
    row = count_features[doc_id].tocoo()
    pairs = ','.join(
        f"{col}:{val}"
        for col, val in sorted(zip(row.col, row.data))
    )
    out_file.write(f"#{doc_id},{pairs}\n")
out_file.close()

In [22]:
with open('count_vectors.txt') as f:
    lines = f.read().splitlines()

In [23]:
len(lines)

19662

In [24]:
lines[0]

'#0,687:1,1028:1,1716:1,1792:1,2289:1,2481:1,2602:1,2892:2,3010:1,3087:1,3193:1,3258:1,3549:2,3552:1,3832:1,3934:1,4224:2,4234:1,4427:1,4639:2,5260:1,5668:1,6726:1,7092:1,7207:1,7406:1,7520:1,7522:1'

In [25]:
tVectorizer = TfidfVectorizer(analyzer="word", vocabulary=vocab)
tfidf_features = tVectorizer.fit_transform(joined_reviews)

In [26]:
tfidf_features.shape

(19662, 7529)

In [27]:
sample = tfidf_features[0].tocoo()
top = sorted(zip(sample.col, sample.data), key=lambda x: -x[1])[:8]

print("Top TF-IDF words in review 0:")
for index, score in top:
    print(f"  '{index_to_word[int(index)]}': {score:.4f}")

Top TF-IDF words in review 0:
  'net': 0.4868
  'half': 0.2999
  'layer': 0.2739
  'outrageously': 0.2568
  'directly': 0.2204
  'reordered': 0.2012
  'major': 0.1947
  'imo': 0.1931


In [ ]:
embedding_model = api.load('glove-wiki-gigaword-100')

In [ ]:
embedding_dim = embedding_model.vector_size
print('Embedding dimension:', embedding_dim)

In [ ]:
def getDocVec_unweighted(review, model, vec_size):
    vectors = []
    for w in review:
        if w in model:
            vectors.append(model[w])
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(vec_size)

In [ ]:
unweighted_doc_vectors = np.array(
    [getDocVec_unweighted(review, embedding_model, embedding_dim) 
     for review in tk_reviews]
)
unweighted_doc_vectors.shape

In [ ]:
unweighted_doc_vectors[0]

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    analyzer="word",
    vocabulary=vocab
)

tfidf_features = tfidf_vectorizer.fit_transform(joined_reviews)

print("TF-IDF matrix shape:", tfidf_features.shape)

In [ ]:
def getDocVec_weighted(row_index, review, model, tfidf_matrix, vocab, embedding_dim):
   
    words = str(review).split()

    weighted_vector = np.zeros(embedding_dim)
    total_weight    = 0

    for word in words:
        if word in model and word in vocab:
            vocab_index = vocab[word]                           # get word index from vocab dict
            weight      = tfidf_matrix[row_index, vocab_index] # TF-IDF score for this word in this doc

            weighted_vector += model[word] * weight
            total_weight    += weight

    if total_weight == 0:
        return np.zeros(embedding_dim)

    return weighted_vector / total_weight

In [ ]:
weighted_doc_vectors = np.array([
    getDocVec_weighted(i, review, embedding_model, tfidf_features, vocab, embedding_dim)
    for i, review in enumerate(df['Review Text'])
])

print("Weighted vectors shape:", weighted_doc_vectors.shape)

In [ ]:
# Double check
weighted_doc_vectors[0]

In [ ]:
print("Unweighted (review 0):", unweighted_doc_vectors[0][:8])
print("Weighted   (review 0):", weighted_doc_vectors[0][:8])

### Saving outputs
Save the count vector representation as per spectification.
- count_vectors.txt

In [ ]:
# code to save output data...

## Task 3. Clothing Review Classification

...... Sections and code blocks on buidling classification models based on different document feature represetations. 
Detailed comparsions and evaluations on different models to answer each question as per specification. 

<span style="color: red"> You might have complex notebook structure in this section, please feel free to create your own notebook structure. </span>

In [ ]:
# Code to perform the task...


## Summary
Give a short summary and anything you would like to talk about the assessment tasks here.

## Couple of notes for all code blocks in this notebook
- please provide proper comment on your code
- Please re-start and run all cells to make sure codes are runable and include your output in the submission.   
<span style="color: red"> This markdown block can be removed once the task is completed. </span>